# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nihaaarika/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row = one user-item interaction (one time a user was shown a content item).

**Time Window:** March 2026 (2026-03). I am using this mid-panel month for development. I will not use the final month (June 2026) to avoid label leakage.

) Data contract
1. What does one row mean?
One row represents one client × content item × report date observation in the daily performance warehouse. The February source data was verified to have no duplicate combinations of these three fields.

2. Which table(s) will you use?
I use the fact_content_daily_performance table for daily GSC performance data and the content dimension table for content metadata such as publication/creation information.

3. What time window will you use?
I use February 2026 (2026-02-01 to 2026-02-28) as the feature window. March 2026 is used as the future outcome window for the label.

4. What will you predict or rank?
I will predict whether a content item goes dark in March 2026, using the binary label went_dark. A value of 1 means the content had zero measured GSC clicks during March; a value of 0 means it had at least one measured March GSC day and did not go dark.

5. What will you deliberately exclude?
I deliberately exclude March performance variables such as imp_mar, clk_mar, and measured_days_mar from the predictive features because they belong to the future outcome window and would not be known at the February 28 decision moment.

In [3]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

def get_hf_token():
    """Get Hugging Face READ token without hard-coding it."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok

    for candidate in (".env", "../.env", "../../.env"):
        if os.path.exists(candidate):
            with open(candidate) as fh:
                for line in fh:
                    if line.startswith("HF_TOKEN="):
                        return line.split("=", 1)[1].strip()

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the token out of SQL text.
hf_token = get_hf_token()
con.execute("SET VARIABLE hf_token = ?", [hf_token])
con.execute("""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN getvariable('hf_token'))
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("connected; feature window = Feb 2026, label window = Mar 2026")


connected; feature window = Feb 2026, label window = Mar 2026


In [4]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Token starts with hf_:", token.startswith("hf_") if token else False)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from huggingface_hub import whoami

info = whoami(token=token)
print("Logged in as:", info["name"])

In [ ]:
from huggingface_hub import hf_hub_download

test_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    token=token
)

print("File downloaded successfully!")
print(test_file)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:** 
- `time_of_day`: Hour of day (0-23)
- `device_type`: Mobile, desktop, tablet
- `user_historical_ctr`: User's past click-through rate

**Label:** 
- `clicked`: 1 if user clicked, 0 if not

**Context:** 
- `date`: When the interaction happened
- `user_id`: Who the user is
- `item_id`: What content was shown

**Excluded:** 
- `session_id`: Excluded because it is unique to each session and would cause overfitting. It does not generalize to new data.

**1. Unit of Analysis:** One row = one content item's daily performance for a specific client.
**2. Tables Used:** `fact_content_daily_performance`
**3. Time Window:** March 2026 (2026-03). I will not use June 2026 to avoid label leakage.
**4. Label/Proxy:** Predicting whether the content will be clicked (using `clicks > 0` as a proxy for engagement).
**5. Deliberately Excluded:** I will exclude June 2026 (the sealed test month) and rows with no impressions.

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW feb_agg AS
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions_feb,
    SUM(gsc_clicks) AS gsc_clicks_feb,
    SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_feb,
    COUNT(*) FILTER (WHERE gsc_data_available) AS measured_days_feb
FROM {FEB}
GROUP BY client_hash_id, content_hash_id
HAVING
    SUM(gsc_impressions) >= 100
    AND SUM(gsc_clicks) >= 3
""")

universe = con.sql(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions_feb,
    f.gsc_clicks_feb,
    f.avg_position_feb,
    f.measured_days_feb,
    d.content_created_date,
    d.is_published
FROM feb_agg f
JOIN {DIM} d
  ON f.content_hash_id = d.content_hash_id
WHERE d.is_published IS TRUE
  AND d.content_created_date <= DATE '2026-02-28'
""").df()

universe["content_created_date"] = pd.to_datetime(
    universe["content_created_date"]
)

universe["ctr_feb"] = (
    universe["gsc_clicks_feb"] /
    universe["gsc_impressions_feb"]
)

universe["content_age_days"] = (
    pd.Timestamp("2026-02-28") -
    universe["content_created_date"]
).dt.days

print(f"February universe rows: {len(universe):,}")
print(f"clients: {universe['client_hash_id'].nunique():,}")


=== FACT 1: GRAIN ===


NameError: name 'df_march' is not defined

Deliberate leakage experiment
I intentionally create one invalid feature derived directly from the label to demonstrate target leakage.

The invalid feature is leak_went_dark, which is simply a copy of went_dark. This information would not be available at the February 28 decision moment because it describes the March outcome.

If this feature is given to a classifier, the model can recover the target directly, so the quick score should become artificially close to perfect.

This is not a valid model result. It demonstrates why a feature must be available at the decision moment and must not be derived from the outcome.

In [ ]:
universe = con.sql(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions_feb,
    f.gsc_clicks_feb,
    f.avg_position_feb,
    f.measured_days_feb,
    d.content_created_date,
    d.is_published
FROM feb_agg f
JOIN {DIM} d
  ON f.content_hash_id = d.content_hash_id
WHERE d.is_published IS TRUE
  AND d.content_created_date <= DATE '2026-02-28'
""").df()

universe["content_created_date"] = pd.to_datetime(
    universe["content_created_date"]
)

universe["ctr_feb"] = (
    universe["gsc_clicks_feb"] /
    universe["gsc_impressions_feb"]
)

universe["content_age_days"] = (
    pd.Timestamp("2026-02-28") -
    universe["content_created_date"]
).dt.days

print(f"February universe rows: {len(universe):,}")
print(f"clients: {universe['client_hash_id'].nunique():,}")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Feature 1: day_of_week** - Knowable at the decision moment because the calendar date is known in advance.
**Feature 2: client_historical_ctr** - Knowable at the decision moment because it uses only past performance data available in the warehouse.
**Feature 3: content_popularity** - Knowable at the decision moment because it aggregates past clicks already recorded.
**Feature 4: is_weekend** - Knowable at the decision moment because it is derived directly from the known date.
**Feature 5: prev_day_clicks** - Knowable at the decision moment because yesterday's data is already finalized and available.

**Feature 1: day_of_week** - Knowable at the decision moment because the calendar date is known in advance.
**Feature 2: client_historical_ctr** - Knowable at the decision moment because it uses only past performance data available in the warehouse.
**Feature 3: content_popularity** - Knowable at the decision moment because it aggregates past clicks already recorded.
**Feature 4: is_weekend** - Knowable at the decision moment because it is derived directly from the known date.
**Feature 5: prev_day_clicks** - Knowable at the decision moment because yesterday's data is already finalized and available.

In [ ]:
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS duplicate_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id
    FROM {FEB}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
""").df()

grain_check

NameError: name 'df_march' is not defined

In [ ]:
con.sql(f"""
SELECT *
FROM {FEB}
LIMIT 5
""").df()

In [ ]:
window_check = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {FEB}
""").df()

window_check

In [ ]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
FROM {FEB}
""").df()

availability_check

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**The Leak Trap:** I will create a label-derived column on purpose. This column uses future information (the next session's click) which is not available at the decision moment. I will show how the score jumps to near-perfect, then delete it to get the honest number.

In [ ]:
features = universe[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_feb",
        "gsc_clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "content_age_days"
    ]
].copy()

features.head()

NameError: name 'df_march' is not defined

### Feature availability

| Feature | Why is it available at the decision moment? |
|---|---|
| `gsc_impressions_feb` | Available by February 28 because it is calculated only from Google Search Console impressions observed during February. |
| `gsc_clicks_feb` | Available by February 28 because it is calculated only from clicks observed during the February feature window. |
| `ctr_feb` | Available by February 28 because it is calculated from February clicks and February impressions only. |
| `avg_position_feb` | Available by February 28 because it is calculated only from search-position observations in the February feature window. |
| `content_age_days` | Available by February 28 because the content creation date was already known at the decision moment. |

In [ ]:
label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS imp_mar,
    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clk_mar,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_days_mar
FROM {MAR}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print(label.head())

In [ ]:
frame = universe.merge(
    label,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print(frame.shape)
frame.head()

In [ ]:
frame["went_dark"] = (
    (frame["measured_days_mar"] > 0) &
    (frame["clk_mar"].fillna(0) == 0)
).astype(int)

print(frame["went_dark"].value_counts())
print()
print("Went-dark rate:", round(frame["went_dark"].mean(), 4))

### Label definition

`went_dark = 1` when a content item has zero measured GSC clicks during March 2026. Otherwise, `went_dark = 0`.

The March label is calculated only after the February feature window. Therefore, March performance is treated as the future outcome and is not used in the five predictive features.

In this dataset, 1,159 of 29,700 content items went dark, giving a went-dark rate of 3.9%.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

FINAL_FEATURES = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
]

model_data = frame[
    FINAL_FEATURES + ["went_dark"]
].dropna().copy()

X = model_data[FINAL_FEATURES]
y = model_data["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)